<a href="https://colab.research.google.com/github/ashikjoel/-Context-Aware-Neural-Recommendation-Engine/blob/ashik_joel/model_training_week_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q tensorflow-recommenders==0.7.7 tf-keras==2.20.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 4.5 MB/s eta 0:00:00


In [1]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tensorflow_recommenders as tfrs

print("TensorFlow:", tf.__version__)
print("TFRS:", tfrs.__version__)

TensorFlow: 2.20.0
TFRS: v0.7.7


In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
BASE_PATH = "/content/drive/MyDrive/Recommendation_Engine"

DATA_PATH = f"{BASE_PATH}/data/processed"
FEATURE_PATH = f"{BASE_PATH}/features"
VOCAB_PATH = f"{BASE_PATH}/vocabularies"
MODEL_PATH = f"{BASE_PATH}/models"

print("Base path:", BASE_PATH)
print("Data path:", DATA_PATH)
print("Feature path:", FEATURE_PATH)
print("Vocabulary path:", VOCAB_PATH)

Base path: /content/drive/MyDrive/Recommendation_Engine
Data path: /content/drive/MyDrive/Recommendation_Engine/data/processed
Feature path: /content/drive/MyDrive/Recommendation_Engine/features
Vocabulary path: /content/drive/MyDrive/Recommendation_Engine/vocabularies


In [6]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HM Recommendation Week 2")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.3


In [7]:
customer_vocab_df = spark.read.parquet(
    f"{VOCAB_PATH}/customer_vocab.parquet"
)

article_vocab_df = spark.read.parquet(
    f"{VOCAB_PATH}/article_vocab.parquet"
)

print("Customer vocabulary rows:", customer_vocab_df.count())
print("Article vocabulary rows :", article_vocab_df.count())

Customer vocabulary rows: 1371980
Article vocabulary rows : 105542


In [8]:
print("Customer vocabulary columns:")
customer_vocab_df.printSchema()

print("\nArticle vocabulary columns:")
article_vocab_df.printSchema()

Customer vocabulary columns:
root
 |-- customer_id: string (nullable = true)


Article vocabulary columns:
root
 |-- article_id: integer (nullable = true)



In [9]:
print("Customer vocabulary:")
customer_vocab_df.show(5, truncate=False)

print("\nArticle vocabulary:")
article_vocab_df.show(5, truncate=False)

Customer vocabulary:
+----------------------------------------------------------------+
|customer_id                                                     |
+----------------------------------------------------------------+
|7f8b9b0a806fac06c6bac8bd3aa516591560e04f4d7b1b69a46f9a8208400a51|
|7f8b9ef211ea02981d0576657e49720dc4860d096a527d6bd23879688ec533da|
|7f8ba63d7266bec3428e8b00b9e1aa82e89c86a473235495f0047604064ef44d|
|7f8bb6c6bac4db37c686e72db68330e049f7dbd4d9d94f99db8f7e15fad0345b|
|7f8bdbcae9710dec1d5cfd26c8f506b709cb31592b38a3739ab8b390f28b3967|
+----------------------------------------------------------------+
only showing top 5 rows

Article vocabulary:
+----------+
|article_id|
+----------+
|108775015 |
|108775044 |
|108775051 |
|110065001 |
|110065002 |
+----------+
only showing top 5 rows


In [10]:
customer_vocab = tf.constant(
    [row.customer_id for row in customer_vocab_df.collect()],
    dtype=tf.string
)

article_vocab = tf.constant(
    [str(row.article_id) for row in article_vocab_df.collect()],
    dtype=tf.string
)

print("Customer vocabulary size:", tf.shape(customer_vocab)[0])
print("Article vocabulary size :", tf.shape(article_vocab)[0])

Customer vocabulary size: tf.Tensor(1371980, shape=(), dtype=int32)
Article vocabulary size : tf.Tensor(105542, shape=(), dtype=int32)


In [11]:
customer_lookup = tf.keras.layers.StringLookup(
    vocabulary=customer_vocab,
    mask_token=None
)

article_lookup = tf.keras.layers.StringLookup(
    vocabulary=article_vocab,
    mask_token=None
)

print("Customer lookup size:", customer_lookup.vocabulary_size())
print("Article lookup size :", article_lookup.vocabulary_size())

Customer lookup size: 1371981
Article lookup size : 105543


In [12]:
EMBEDDING_DIM = 64

customer_embedding = tf.keras.layers.Embedding(
    input_dim=customer_lookup.vocabulary_size(),
    output_dim=EMBEDDING_DIM
)

article_embedding = tf.keras.layers.Embedding(
    input_dim=article_lookup.vocabulary_size(),
    output_dim=EMBEDDING_DIM
)

print("Customer embedding dimension:", EMBEDDING_DIM)
print("Article embedding dimension :", EMBEDDING_DIM)

Customer embedding dimension: 64
Article embedding dimension : 64


In [13]:
query_tower = tf.keras.Sequential([
    customer_lookup,
    customer_embedding
], name="query_tower")

print(query_tower)

In [14]:
test_customer = customer_vocab[0:1]

user_embedding = query_tower(test_customer)

print("Input shape :", test_customer.shape)
print("Embedding shape:", user_embedding.shape)

Input shape : (1,)
Embedding shape: (1, 64)


In [15]:
candidate_tower = tf.keras.Sequential([
    article_lookup,
    article_embedding
], name="candidate_tower")

print(candidate_tower)

In [16]:
test_article = article_vocab[0:1]

item_embedding = candidate_tower(test_article)

print("Input shape    :", test_article.shape)
print("Embedding shape:", item_embedding.shape)

Input shape    : (1,)
Embedding shape: (1, 64)


In [17]:
print("User Embedding Shape :", query_tower(test_customer).shape)
print("Item Embedding Shape :", candidate_tower(test_article).shape)

User Embedding Shape : (1, 64)
Item Embedding Shape : (1, 64)


In [18]:
transactions = spark.read.parquet(
    f"{DATA_PATH}/transactions_clean.parquet"
)

print("Transaction count:", transactions.count())
transactions.printSchema()

Transaction count: 31788324
root
 |-- t_dat: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)



In [19]:
from pyspark.sql import functions as F

transactions.select(
    F.min("t_dat").alias("earliest_date"),
    F.max("t_dat").alias("latest_date")
).show()

+-------------+-----------+
|earliest_date|latest_date|
+-------------+-----------+
|   2018-09-20| 2020-09-22|
+-------------+-----------+



In [20]:
transactions.groupBy(
    F.year("t_dat").alias("year"),
    F.month("t_dat").alias("month")
).count().orderBy(
    "year", "month"
).show(30)

+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2018|    9| 594776|
|2018|   10|1397040|
|2018|   11|1270619|
|2018|   12|1148827|
|2019|    1|1263471|
|2019|    2|1152412|
|2019|    3|1286750|
|2019|    4|1476454|
|2019|    5|1560319|
|2019|    6|1906202|
|2019|    7|1807494|
|2019|    8|1253530|
|2019|    9|1227178|
|2019|   10|1146772|
|2019|   11|1198033|
|2019|   12|1118315|
|2020|    1|1076354|
|2020|    2|1001859|
|2020|    3|1047752|
|2020|    4|1340882|
|2020|    5|1361815|
|2020|    6|1764507|
|2020|    7|1351502|
|2020|    8|1237192|
|2020|    9| 798269|
+----+-----+-------+



In [21]:
train_transactions = transactions.filter(
    F.col("t_dat") < F.lit("2020-09-16")
)

test_transactions = transactions.filter(
    F.col("t_dat") >= F.lit("2020-09-16")
)

print("Training transactions:", train_transactions.count())
print("Testing transactions :", test_transactions.count())

Training transactions: 31548013
Testing transactions : 240311


In [22]:
train_users = train_transactions.select("customer_id").distinct()
test_users = test_transactions.select("customer_id").distinct()

test_seen_users = test_users.join(
    train_users,
    on="customer_id",
    how="inner"
)

print("Test users:", test_users.count())
print("Test users seen during training:", test_seen_users.count())

Test users: 68984
Test users seen during training: 63412


In [23]:
train_interactions = (
    train_transactions
    .select("customer_id", "article_id")
    .dropDuplicates(["customer_id", "article_id"])
)

print("Unique training interactions:", train_interactions.count())

Unique training interactions: 27101148


In [24]:
train_sample = train_interactions.sample(
    withReplacement=False,
    fraction=1_000_000 / 27_101_148,
    seed=42
)

print("Sampled training interactions:", train_sample.count())

Sampled training interactions: 999664


In [25]:
train_ds = tf.data.Dataset.from_tensor_slices({
    "customer_id": train_sample.select("customer_id").toPandas()["customer_id"].values,
    "article_id": train_sample.select("article_id").toPandas()["article_id"].astype(str).values
})

print("Dataset created successfully!")

Dataset created successfully!


In [26]:
for example in train_ds.take(1):
    print(example)

{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'af81e692945e512383d7dcf12dd0a9d6adcb108916300c2d8b12663b4a966bb6'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'828114001'>}


In [27]:
BATCH_SIZE = 8192

train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Batch size:", BATCH_SIZE)

Batch size: 8192


In [28]:
retrieval_task = tfrs.tasks.Retrieval()

print("TFRS Retrieval task created successfully!")

TFRS Retrieval task created successfully!


In [29]:
class HMRecommendationModel(tfrs.models.Model):

    def __init__(self, query_model, candidate_model):
        super().__init__()
        self.query_model = query_model
        self.candidate_model = candidate_model
        self.task = retrieval_task

    def compute_loss(self, features, training=False):
        query_embeddings = self.query_model(
            features["customer_id"]
        )

        candidate_embeddings = self.candidate_model(
            features["article_id"]
        )

        return self.task(
            query_embeddings,
            candidate_embeddings
        )


model = HMRecommendationModel(
    query_model=query_tower,
    candidate_model=candidate_tower
)

print("Two-Tower TFRS model created successfully!")

Two-Tower TFRS model created successfully!


In [30]:
test_batch = next(iter(train_ds))

loss = model.compute_loss(test_batch)

print("Test batch loaded successfully")
print("Loss:", float(loss))

Test batch loaded successfully
Loss: 73816.75


In [31]:
model.compile(
    optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1)
)

print("Model compiled successfully!")

Model compiled successfully!


In [32]:
CHECKPOINT_DIR = f"{BASE_PATH}/models/week2_checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Checkpoint directory:", CHECKPOINT_DIR)

Checkpoint directory: /content/drive/MyDrive/Recommendation_Engine/models/week2_checkpoints


In [33]:
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(
        CHECKPOINT_DIR,
        "epoch_{epoch:02d}.weights.h5"
    ),
    save_weights_only=True,
    save_freq="epoch",
    verbose=1
)

print("Checkpoint callback created successfully!")

Checkpoint callback created successfully!


In [34]:
print(os.listdir(CHECKPOINT_DIR))

[]


In [35]:
history = model.fit(
    train_ds,
    epochs=1,
    callbacks=[checkpoint_callback],
    verbose=1
)

123/123 [==============================] - ETA: 0s - loss: 73228.7524 - regularization_loss: 0.0000e+00 - total_loss: 73228.7524
Epoch 1: saving model to /content/drive/MyDrive/Recommendation_Engine/models/week2_checkpoints/epoch_01.weights.h5


123/123 [==============================] - 59s 469ms/step - loss: 72648.8055 - regularization_loss: 0.0000e+00 - total_loss: 72648.8055


In [36]:
history_epoch2 = model.fit(
    train_ds,
    epochs=2,
    initial_epoch=1,
    callbacks=[checkpoint_callback],
    verbose=1
)

Epoch 2/2
121/123 [============================>.] - ETA: 0s - loss: 72281.3877 - regularization_loss: 0.0000e+00 - total_loss: 72281.3877
Epoch 2: saving model to /content/drive/MyDrive/Recommendation_Engine/models/week2_checkpoints/epoch_02.weights.h5


123/123 [==============================] - 49s 398ms/step - loss: 71123.5063 - regularization_loss: 0.0000e+00 - total_loss: 71123.5063


In [37]:
history_epoch3 = model.fit(
    train_ds,
    epochs=3,
    initial_epoch=2,
    callbacks=[checkpoint_callback],
    verbose=1
)

Epoch 3/3
122/123 [============================>.] - ETA: 0s - loss: 63338.9793 - regularization_loss: 0.0000e+00 - total_loss: 63338.9793
Epoch 3: saving model to /content/drive/MyDrive/Recommendation_Engine/models/week2_checkpoints/epoch_03.weights.h5


123/123 [==============================] - 50s 405ms/step - loss: 62330.1946 - regularization_loss: 0.0000e+00 - total_loss: 62330.1946


In [38]:
print(os.listdir(CHECKPOINT_DIR))

['epoch_01.weights.h5', 'epoch_02.weights.h5', 'epoch_03.weights.h5']


In [40]:
query_tower.save_weights(
    f"{TRAINED_MODEL_DIR}/query_tower_trained.weights.h5"
)

print("Trained Query Tower weights saved successfully!")

Trained Query Tower weights saved successfully!


In [41]:
candidate_tower.save_weights(
    f"{TRAINED_MODEL_DIR}/candidate_tower_trained.weights.h5"
)

print("Trained Candidate Tower weights saved successfully!")

Trained Candidate Tower weights saved successfully!


In [42]:
print("Training checkpoints:")
print(os.listdir(CHECKPOINT_DIR))

print("\nTrained towers:")
print(os.listdir(TRAINED_MODEL_DIR))

Training checkpoints:
['epoch_01.weights.h5', 'epoch_02.weights.h5', 'epoch_03.weights.h5']

Trained towers:
['query_tower_trained.keras', 'query_tower_trained.weights.h5', 'candidate_tower_trained.weights.h5']


In [43]:
index = tfrs.layers.factorized_top_k.BruteForce(
    query_tower
)

print("Retrieval index created successfully!")

Retrieval index created successfully!


In [44]:
print("Number of unique articles:", article_vocab.shape[0])
print("First 5 article IDs:", article_vocab[:5].numpy())

Number of unique articles: 105542
First 5 article IDs: [b'108775015' b'108775044' b'108775051' b'110065001' b'110065002']


In [45]:
index.index_from_dataset(
    tf.data.Dataset.from_tensor_slices(article_vocab).batch(512).map(
        lambda x: (x, candidate_tower(x))
    )
)

print("Retrieval index built successfully!")

Retrieval index built successfully!


In [46]:
test_customer_id = train_sample.select("customer_id").first()["customer_id"]

scores, recommended_ids = index(
    tf.constant([test_customer_id])
)

print("Customer ID:", test_customer_id)
print("Recommended articles:")
print(recommended_ids[0].numpy())

Customer ID: af81e692945e512383d7dcf12dd0a9d6adcb108916300c2d8b12663b4a966bb6
Recommended articles:
[b'828114001' b'778064005' b'556539003' b'600886001' b'815264002'
 b'736681001' b'591334003' b'796210010' b'542533002' b'680262004']


In [47]:
evaluation_users = test_seen_users

print(
    "Evaluation users:",
    evaluation_users.count()
)

Evaluation users: 63412


In [48]:
evaluation_test = (
    test_transactions
    .join(
        evaluation_users,
        on="customer_id",
        how="inner"
    )
    .select("customer_id", "article_id")
    .dropDuplicates(["customer_id", "article_id"])
)

print("Unique evaluation interactions:", evaluation_test.count())

Unique evaluation interactions: 197513


In [49]:
actual_items_df = (
    evaluation_test
    .groupBy("customer_id")
    .agg(
        F.collect_set("article_id").alias("actual_items")
    )
)

print("Customers with test interactions:", actual_items_df.count())

Customers with test interactions: 63412


In [50]:
evaluation_customer_ids = (
    evaluation_users
    .select("customer_id")
    .rdd
    .map(lambda row: row.customer_id)
    .collect()
)

print("Evaluation customer IDs:", len(evaluation_customer_ids))
print("First customer:", evaluation_customer_ids[0])

Evaluation customer IDs: 63412
First customer: ab1e1d2cfc55578021587acad6e035fa04a53838d417b83db78a5ce2d12be849


In [51]:
test_eval_customers = evaluation_customer_ids[:100]

scores, predictions = index(
    tf.constant(test_eval_customers)
)

print("Prediction shape:", predictions.shape)
print("First customer predictions:")
print(predictions[0].numpy())

Prediction shape: (100, 10)
First customer predictions:
[b'478646001' b'731160005' b'816166002' b'253448003' b'372008001'
 b'684209019' b'751471016' b'751998001' b'662948012' b'578374001']


In [52]:
import numpy as np

all_predictions = []

BATCH_SIZE_EVAL = 500

for start in range(0, len(evaluation_customer_ids), BATCH_SIZE_EVAL):
    batch_customers = evaluation_customer_ids[
        start:start + BATCH_SIZE_EVAL
    ]

    _, batch_predictions = index(
        tf.constant(batch_customers)
    )

    all_predictions.append(batch_predictions.numpy())

    print(
        f"Processed {min(start + BATCH_SIZE_EVAL, len(evaluation_customer_ids))}"
        f"/{len(evaluation_customer_ids)}"
    )

all_predictions = np.concatenate(all_predictions, axis=0)

print("\nFinal prediction shape:", all_predictions.shape)

Processed 500/63412
Processed 1000/63412
Processed 1500/63412
Processed 2000/63412
Processed 2500/63412
Processed 3000/63412
Processed 3500/63412
Processed 4000/63412
Processed 4500/63412
Processed 5000/63412
Processed 5500/63412
Processed 6000/63412
Processed 6500/63412
Processed 7000/63412
Processed 7500/63412
Processed 8000/63412
Processed 8500/63412
Processed 9000/63412
Processed 9500/63412
Processed 10000/63412
Processed 10500/63412
Processed 11000/63412
Processed 11500/63412
Processed 12000/63412
Processed 12500/63412
Processed 13000/63412
Processed 13500/63412
Processed 14000/63412
Processed 14500/63412
Processed 15000/63412
Processed 15500/63412
Processed 16000/63412
Processed 16500/63412
Processed 17000/63412
Processed 17500/63412
Processed 18000/63412
Processed 18500/63412
Processed 19000/63412
Processed 19500/63412
Processed 20000/63412
Processed 20500/63412
Processed 21000/63412
Processed 21500/63412
Processed 22000/63412
Processed 22500/63412
Processed 23000/63412
Processe

In [54]:
# Create a mapping from customer ID → prediction row
prediction_map = {
    customer_id: all_predictions[i]
    for i, customer_id in enumerate(evaluation_customer_ids)
}

# Put predictions into the same order as actual_items_df
ordered_predictions = np.array([
    prediction_map[row.customer_id]
    for row in actual_items_rows
])

print("Ordered predictions shape:", ordered_predictions.shape)

Ordered predictions shape: (63412, 10)


In [55]:
recall_scores = []
ndcg_scores = []

for actual_row, predicted_row in zip(
    actual_items_rows,
    ordered_predictions
):
    actual = set(str(x) for x in actual_row.actual_items)
    predicted = [
        x.decode("utf-8") if isinstance(x, bytes) else str(x)
        for x in predicted_row
    ]

    # Recall@10
    hits = len(set(predicted) & actual)
    recall = hits / len(actual) if actual else 0.0
    recall_scores.append(recall)

    # NDCG@10
    dcg = 0.0

    for rank, item in enumerate(predicted, start=1):
        if item in actual:
            dcg += 1.0 / np.log2(rank + 1)

    ideal_hits = min(len(actual), 10)

    idcg = sum(
        1.0 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    ndcg = dcg / idcg if idcg > 0 else 0.0
    ndcg_scores.append(ndcg)

print("Recall@10:", np.mean(recall_scores))
print("NDCG@10:", np.mean(ndcg_scores))

Recall@10: 0.002000116941543045
NDCG@10: 0.0016158549899144095


In [ ]:
unique_recommendations = np.unique(all_predictions)

print("Total recommendations:", all_predictions.size)
print("Unique recommended articles:", len(unique_recommendations))
print(
    "Recommendation diversity:",
    len(unique_recommendations) / all_predictions.size
)